# Motor de Voz Ultra Realista (ChatTTS) para Miniverse
Este cuaderno permite correr **ChatTTS** en Google Colab y exponerlo mediante un túnel (ngrok) para conectarlo a tu backend local.

### ⚠️ PASO MUY IMPORTANTE (NO OMITIR):
Asegúrate de ir al menú superior: **Entorno de Ejecución > Cambiar tipo de entorno de ejecución** y elige **T4 GPU**. (Si no lo haces, correrá en CPU y será lentísimo).


In [ ]:
!pip install fastapi uvicorn pyngrok soundfile numpy gTTS chatterbox-tts==0.1.7 torchaudio>=2.0.0


### Crear el servidor API
Esta celda guarda el código en un archivo interno para evitar errores de bucles de ejecución.

In [ ]:
%%writefile app.py
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
import warnings
warnings.filterwarnings('ignore')

import os
import re
import urllib.request
import torch
import numpy as np
from chatterbox.tts import ChatterboxTTS

# Prepare directories and reference Spanish voice WAVs
os.makedirs("ref_voices", exist_ok=True)
male_wav_path = "ref_voices/male.wav"
female_wav_path = "ref_voices/female.wav"

# Download male reference Spanish voice from sherpa-onnx if not present
if not os.path.exists(male_wav_path):
    try:
        print("Descargando referencia de voz masculina en español...")
        urllib.request.urlretrieve(
            "https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/es.wav", 
            male_wav_path
        )
        print("Referencia masculina descargada.")
    except Exception as e:
        print("Error al descargar referencia de voz masculina:", e)

# Generate female reference Spanish voice using gTTS if not present
if not os.path.exists(female_wav_path):
    try:
        print("Generando referencia de voz femenina en español...")
        from gtts import gTTS
        tts = gTTS(text="Hola, soy Valeria. Estoy lista para avanzar con las tareas y coordinar el desarrollo del software.", lang="es")
        tts.save("ref_voices/female.mp3")
        os.system("ffmpeg -y -i ref_voices/female.mp3 -acodec pcm_s16le -ac 1 -ar 24000 ref_voices/female.wav > /dev/null 2>&1")
        print("Referencia femenina generada.")
    except Exception as e:
        print("Error al generar referencia de voz femenina:", e)

app = FastAPI()
print("Cargando modelo Chatterbox ultra realista en GPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ChatterboxTTS.from_pretrained(device=device)
print(f"Modelo cargado en dispositivo: {device}")

class VoiceRequest(BaseModel):
    text: str
    sexo: str = "neutral"
    language: str = "es"

def get_wav_header(sample_rate=24000, num_channels=1, bits_per_sample=16):
    # Standard 44-byte WAV header with maxed out data chunk size for infinite/streaming audio
    data_size = 0x7FFFFFFF
    file_size = data_size + 36
    
    header = bytearray()
    header.extend(b'RIFF')
    header.extend(file_size.to_bytes(4, byteorder='little'))
    header.extend(b'WAVE')
    header.extend(b'fmt ')
    header.extend((16).to_bytes(4, byteorder='little')) # Subchunk1Size
    header.extend((1).to_bytes(2, byteorder='little'))  # AudioFormat (1 = PCM)
    header.extend(num_channels.to_bytes(2, byteorder='little'))
    header.extend(sample_rate.to_bytes(4, byteorder='little'))
    
    byte_rate = sample_rate * num_channels * (bits_per_sample // 8)
    block_align = num_channels * (bits_per_sample // 8)
    
    header.extend(byte_rate.to_bytes(4, byteorder='little'))
    header.extend(block_align.to_bytes(2, byteorder='little'))
    header.extend(bits_per_sample.to_bytes(2, byteorder='little'))
    header.extend(b'data')
    header.extend(data_size.to_bytes(4, byteorder='little'))
    return bytes(header)

def split_text(text: str):
    # Splits text by standard end of sentence punctuation
    sentences = re.split(r'(?<=[.!?;\n])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

def generate_audio_stream(text: str, sexo: str):
    sentences = split_text(text)
    
    # 1. Yield the 44-byte WAV header first
    sample_rate = model.sr if hasattr(model, 'sr') else 24000
    yield get_wav_header(sample_rate=sample_rate)
    
    # Choose voice prompt path based on gender
    spk_path = female_wav_path if sexo.lower() == "femenino" else male_wav_path
    kwargs = {"audio_prompt_path": spk_path} if os.path.exists(spk_path) else {}
    
    # 2. Yield raw 16-bit PCM bytes for each sentence as it completes
    for sentence in sentences:
        if not sentence.strip():
            continue
        try:
            print(f"Sintetizando oración: '{sentence}'")
            # Generate float waveform
            wav = model.generate(sentence, **kwargs)
            wav = wav.squeeze()
            
            # Convert float waveform [-1.0, 1.0] to int16 PCM [-32768, 32767]
            wav_int16 = (wav * 32767.0).clamp(-32768.0, 32767.0).to(torch.int16)
            pcm_bytes = wav_int16.cpu().numpy().tobytes()
            yield pcm_bytes
        except Exception as e:
            print(f"Error generando audio para: '{sentence}': {e}")

@app.post("/synthesize")
def synthesize(req: VoiceRequest):
    print(f"Petición de síntesis en tiempo real recibida para: {req.text[:60]}...")
    return StreamingResponse(
        generate_audio_stream(req.text, req.sexo),
        media_type="audio/wav"
    )


### Iniciar Servidor y Ngrok
Pon tu token de Ngrok, ejecuta esta celda y pon la URL en tu `.env` local.

In [ ]:
import os
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "TU_NGROK_AUTH_TOKEN_AQUI" # <--- PON TU TOKEN AQUÍ
os.environ["NGROK_AUTH_TOKEN"] = NGROK_AUTH_TOKEN
!ngrok config add-authtoken $NGROK_AUTH_TOKEN

public_url = ngrok.connect(8000).public_url
print("\n" + "="*60)
print(f"API PUBLICA ACTIVA EN: {public_url}")
print("Pon esta URL en tu .env local: CHATTERBOX_REMOTE_URL=", public_url)
print("="*60 + "\n")

!uvicorn app:app --host 0.0.0.0 --port 8000